# 13 · Explain predictions and run a saved model

**Level:** advanced · **Prerequisites:** 04, 05, 09, 10 · **Study time:** 90 minutes

You will train a small CNN, visualize input gradients and Grad-CAM,
export weights with their preprocessing contract, and make predictions
from a separate Python script. Everything runs independently of earlier notebooks.

A heatmap indicates a model sensitivity or a spatial association. It does
not prove causal reasoning, fairness, or correctness. Our 8×8 images and
small feature maps are especially coarse; treat the visualizations as
debugging tools and compare them with actual errors.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from course_utils import TinyCNN, get_digits_loaders, fit, evaluate
train_loader, val_loader, test_loader = get_digits_loaders()
model = TinyCNN().to(device)
history = fit(model, train_loader, val_loader,
              epochs=1 if SMOKE_TEST else 12, device=device)


## 1. Pick a validation image and explain its predicted class

Use validation examples while developing explanations. The test set
stays untouched here. The model returns **logits**. We apply softmax
only for displayed probabilities, not before cross-entropy training.


In [ ]:
val_images, val_labels = next(iter(val_loader))
example_index = 0
x = val_images[example_index:example_index + 1].to(device)
model.eval()
with torch.no_grad():
    probabilities = model(x).softmax(1)
predicted_class = int(probabilities.argmax(1).item())
print("True:", int(val_labels[example_index]), "Predicted:", predicted_class)
print("Probability of predicted class:", float(probabilities[0, predicted_class]))
plt.imshow(x[0, 0].cpu(), cmap="gray")
plt.title("Validation example")
plt.show()


## 2. Input-gradient saliency

Ask: if a pixel changed slightly, how much would the chosen class score
change? Enable gradients on the input, backpropagate one **logit**, and
inspect the magnitude of the input gradient. Do not put this inside
`no_grad()` or `inference_mode()`; those intentionally disable gradients.


In [ ]:
saliency_input = x.detach().clone().requires_grad_(True)
model.zero_grad(set_to_none=True)
class_score = model(saliency_input)[0, predicted_class]
class_score.backward()
saliency = saliency_input.grad.detach().abs()[0, 0].cpu()
saliency = saliency / saliency.max().clamp_min(1e-8)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(x[0, 0].cpu(), cmap="gray")
axes[0].set_title("Input")
axes[1].imshow(saliency, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Absolute input gradient")
plt.tight_layout()
plt.show()


## 3. Grad-CAM from the final convolution

A forward hook captures feature maps `A[k]`. Gradients of a selected
logit with respect to each map give channel importance weights:
`alpha[k] = mean(d score / d A[k])` over spatial positions.
Compute `ReLU(sum(alpha[k] * A[k]))`, then resize it for display.
We remove the hook in a `finally` block so rerunning the cell cannot stack hooks.


In [ ]:
def grad_cam(model, image, target_class):
    captured = {}
    def capture(module, inputs, output):
        captured["features"] = output
        output.retain_grad()
    handle = model.features[3].register_forward_hook(capture)
    try:
        model.zero_grad(set_to_none=True)
        score = model(image)[0, target_class]
        score.backward()
        features = captured["features"]
        weights = features.grad.mean(dim=(2, 3), keepdim=True)
        heatmap = (weights * features).sum(dim=1, keepdim=True).relu()
        heatmap = torch.nn.functional.interpolate(
            heatmap, size=image.shape[-2:], mode="bilinear", align_corners=False)
        heatmap = heatmap[0, 0].detach().cpu()
        return heatmap / heatmap.max().clamp_min(1e-8)
    finally:
        handle.remove()

cam = grad_cam(model, x, predicted_class)
plt.imshow(x[0, 0].cpu(), cmap="gray")
plt.imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
plt.title(f"Grad-CAM for class {predicted_class}")
plt.colorbar(label="Relative intensity")
plt.show()


A blank map can occur when the weighted evidence is nonpositive before
ReLU. It is not an error to hide by inventing a heatmap. Try another
image or class and report what actually happens. Colormaps are normalized
per image here, so their brightness cannot be compared as absolute confidence.

## 4. Probe a small distribution shift

Add noise to the same validation batch and compare accuracy. This is one
narrow corruption test, not a robustness certificate. We clamp to the
legal [0,1] input range and keep labels unchanged.


In [ ]:
model.eval()
with torch.no_grad():
    batch = val_images.to(device)
    labels = val_labels.to(device)
    noisy = (batch + 0.20 * torch.randn_like(batch)).clamp(0, 1)
    clean_accuracy = (model(batch).argmax(1) == labels).float().mean().item()
    noisy_accuracy = (model(noisy).argmax(1) == labels).float().mean().item()
print(f"One validation batch: clean={clean_accuracy:.1%}, noisy={noisy_accuracy:.1%}")


## 5. Save a complete inference contract

Weights alone do not specify class order, color conversion, size, or
normalization. Save those details together. This bundle uses only tensors
and simple Python values and loads with `weights_only=True`. Load files
you created or trust. Keep the `TinyCNN` architecture code with the bundle.


In [ ]:
checkpoint_path = ARTIFACTS / "13_digit_classifier.pt"
bundle = {
    "architecture": "TinyCNN-v1",
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "classes": [str(i) for i in range(10)],
    "image_size": 8,
    "channels": 1,
    "preprocessing": "grayscale, bilinear resize to 8x8, divide uint8 pixels by 255",
    "expected_input": "centered light digit on dark background",
    "training_data": "sklearn digits; original pixels divided by 16",
    "seed": 42,
    "smoke_test": SMOKE_TEST,
}
torch.save(bundle, checkpoint_path)
print("Saved:", checkpoint_path)


## 6. Verify the save/load round trip

Compare identical inputs on the same device before and after loading.
Saving should preserve logits. This check catches architecture or weight
mistakes; it does not measure classification quality.


In [ ]:
from predict import load_bundle, predict_image, image_to_tensor
restored, restored_bundle = load_bundle(checkpoint_path, device="cpu")
original_cpu = copy.deepcopy(model).cpu().eval()
probe = val_images[:4].cpu()
with torch.inference_mode():
    before = original_cpu(probe)
    after = restored(probe)
torch.testing.assert_close(before, after)
print("Round-trip logits match.")


## 7. Classify an actual PNG through the inference function

Export one validation digit as an image, then load it using Pillow. The
conversion to uint8 introduces a little quantization, so exact logits may
differ from the original floating point sample. Phone photos are a much
larger domain shift: this tiny digit model is not a general photo classifier.


In [ ]:
demo_path = ARTIFACTS / "13_demo_digit.png"
png_pixels = (val_images[example_index, 0].numpy() * 255).round().astype(np.uint8)
Image.fromarray(png_pixels).save(demo_path)
top_three = predict_image(restored, restored_bundle, demo_path, top_k=3)
print(json.dumps(top_three, indent=2))
print("Image:", demo_path)


From PowerShell in the project folder, the same prediction works without
Jupyter:

```powershell
.\.venv\Scripts\python.exe predict.py artifacts\13_demo_digit.png
```

For your own **dark digit on a light background**, add `--invert`.
Crop and center the digit first. A high softmax value is not a guarantee
of correctness and does not detect unrelated images automatically.


## 8. Measure inference latency honestly

Warm up the model, time repeated forward passes, and state what the timer
excludes. This CPU measurement excludes image decoding, disk I/O, and
network serving. For GPU timing, synchronize CUDA before and after timing;
GPU operations are asynchronous.


In [ ]:
input_tensor = image_to_tensor(demo_path)
repeats = 10 if SMOKE_TEST else 100
with torch.inference_mode():
    for _ in range(5):
        restored(input_tensor)
    started = time.perf_counter()
    for _ in range(repeats):
        restored(input_tensor)
milliseconds = 1000 * (time.perf_counter() - started) / repeats
print(f"CPU forward-only mean latency: {milliseconds:.3f} ms/image")


## Practice: turn explanations into experiments

**A.** Explain the true class and predicted class for a misclassified
validation image. Does the map change? Record the class used for each map.

**B.** Compare noise strengths 0.0, 0.1, and 0.3 on validation data. Keep
a fixed noise realization while changing strength for a fair comparison.

**C.** Export a second image and run `predict.py`. Try an upside-down
version and explain why the confidence may still be high.


In [ ]:
# Exercise A starter: locate mistakes without touching test data.
validation = evaluate(model, val_loader, device)
mistake_indices = (validation["logits"].argmax(1) != validation["targets"]).nonzero().flatten()
print("First validation mistake indices:", mistake_indices[:10].tolist())


In [ ]:
# Exercise B starter: extend the strength list, then compare accuracies.
fixed_noise = torch.randn_like(val_images)
for strength in [0.0, 0.1, 0.3]:
    corrupted = (val_images + strength * fixed_noise).clamp(0, 1).to(device)
    with torch.no_grad():
        accuracy = (model(corrupted).argmax(1).cpu() == val_labels).float().mean().item()
    print(strength, round(accuracy, 3))


<details><summary>Sample solution guidance</summary>

Use `val_loader.dataset[index]` for a mistake index, because the index can
extend beyond the first batch. Pass `image.unsqueeze(0).to(device)` to
`grad_cam` so the batch dimension and device both match the model.
Pass the actual label as `target_class` to explain the true class instead.
Larger noise generally hurts performance, but a small batch need not be
monotonic. Rotating an image changes the distribution; softmax must still
allocate its probability mass among the ten known digits.
</details>

**You can now:** debug a prediction, describe the limits of an explanation,
export a reproducible inference contract, and predict from a separate script.

**References:** [Grad-CAM paper](https://arxiv.org/abs/1610.02391),
[PyTorch saving/loading](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

**Next:** [14 · Your capstone project](14_capstone_project.ipynb).
